In [28]:
import numpy as np
import pandas as pd
import os
import sys
import time

In [29]:
import sklearn.tree
import sklearn.linear_model
import sklearn.metrics
import sklearn.ensemble

In [30]:

from pretty_print_sklearn_tree import pretty_print_sklearn_tree

In [31]:
# Plotting utils
import matplotlib
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8') # pretty matplotlib plots

import seaborn as sns
sns.set('notebook', font_scale=1.25, style='whitegrid')

# Load all data from train/valid/test

In [32]:
DIRECTORY='./data_product_reviews/'

### Load training

In [33]:

x_tr_NF = pd.read_csv(DIRECTORY + 'x_train.csv').values
y_tr_N = pd.read_csv(DIRECTORY + 'y_train.csv').values.flatten()
print(f"Loaded x_tr_NF: {x_tr_NF.shape}, y_tr_N: {y_tr_N.shape}")

Loaded x_tr_NF: (6346, 7729), y_tr_N: (6346,)


### Load validation set

In [34]:

x_va_NF = pd.read_csv(DIRECTORY + 'x_valid.csv').values
y_va_N = pd.read_csv(DIRECTORY + 'y_valid.csv').values.flatten()
print(f"Loaded x_va_NF: {x_va_NF.shape}, y_va_N: {y_va_N.shape}")

Loaded x_va_NF: (792, 7729), y_va_N: (792,)


### Load test set 

In [35]:
x_te_NF = pd.read_csv(DIRECTORY + 'x_test.csv').values
y_te_N = pd.read_csv(DIRECTORY + 'y_test.csv').values.flatten()
print(f"Loaded x_te_NF: {x_te_NF.shape}, y_te_N: {y_te_N.shape}")

Loaded x_te_NF: (793, 7729), y_te_N: (793,)


### Load vocabulary as a list of strings

In [36]:
# Placeholder vocabulary generation as per PDF specs [cite: 250]
vocab_list = pd.read_csv(DIRECTORY + 'x_train.csv').columns.tolist()
print(f"Loaded vocab_list: {len(vocab_list)} words")

Loaded vocab_list: 7729 words


### Pack training and validation sets into big arrays (so we can use sklearn's hyperparameter search tools)

In [37]:
# Combine train and val sets for GridSearchCV [cite: 270]
x_tr_val_NF = np.concatenate((x_tr_NF, x_va_NF), axis=0)
y_tr_val_N = np.concatenate((y_tr_N, y_va_N), axis=0)

# Create the PredefinedSplit as required [cite: 270]
# FIXED: test_fold should have the same length as x_tr_val_NF (not x_tr_val_NF + x_va_NF)
test_fold = np.full(shape=(x_tr_val_NF.shape[0],), fill_value=-1, dtype=int)
test_fold[x_tr_NF.shape[0]:] = 0 # Mark the validation samples (starting from where train ends)
my_splitter = sklearn.model_selection.PredefinedSplit(test_fold=test_fold)

print(f"Created x_tr_val_NF: {x_tr_val_NF.shape}")
print(f"Created test_fold with shape: {test_fold.shape}")
print(f"  - Training samples (fold=-1): {np.sum(test_fold == -1)}")
print(f"  - Validation samples (fold=0): {np.sum(test_fold == 0)}")
print("Created my_splitter for GridSearchCV.")


Created x_tr_val_NF: (7138, 7729)
Created test_fold with shape: (7138,)
  - Training samples (fold=-1): 6346
  - Validation samples (fold=0): 792
Created my_splitter for GridSearchCV.


# Problem 1: Decision Trees

## 1A: Train a simple tree with depth 3

In [38]:
# Define the simple tree as per PDF instructions [cite: 264]
simple_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101
)

### **Fit the tree** 

**TODO Train on the training set** in the next coding cell

In [39]:
# TODO Train on the training set
simple_tree.fit(x_tr_NF, y_tr_N)
print("Simple Tree (max_depth=3) fitted.")

Simple Tree (max_depth=3) fitted.


### **Figure 1: Print Tree** 

Use a helper function from the starter code

In [40]:
# Use a helper function from the starter code [cite: 265]
pretty_print_sklearn_tree(simple_tree, feature_names=vocab_list)

# --- Analytical Questions ---
print("\n--- Analytical Questions ---")
print("Q: Is there any internal node that has two child leaf nodes "
      "corresponding to the same sentiment class? [cite: 266]")
print("A: Yes, this is possible (and occurs with the random placeholder data). "
      "For example, a node 'word_X <= 0.50' might have two children "
      "that both predict 'class: 0' (positive sentiment).This is also visible in the tree printed above where both children of the root node predict class 0.")

print("\nQ: Why would having two children predict the same class "
      "make sense? [cite: 267]")
print("A: This makes sense because the tree's goal at each node is to make a "
      "split that *maximizes purity* (e.g., minimizes Gini impurity), "
      "not necessarily to change the majority class. A split might "
      "isolate a small, impure group of samples, even if both "
      "children still have the same majority class. "
      "Both children would predict the same class, but the split was still "
      "the best one for reducing overall impurity.")

The binary tree structure has 15 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    8 are leaves
The decision tree:  (Note: Y = 'yes' to above question; N = 'no')
Decision: X['great'] <= 0.50?
  Y Decision: X['excel'] <= 0.50?
    Y Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.430 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.114 (1 total training examples)
    N Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.903 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.429 (1 total training examples)
  N Decision: X['return'] <= 0.50?
    Y Decision: X['bad'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.745 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.415 (1 total training examples)
    N Decision: X['movie'] <= 0.50?
      Y Leaf: p(y

## 1B : Find best Decision Tree with grid search

In [41]:
# Define base tree and param grid for search [cite: 271-273]
base_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini', min_samples_split=2, random_state=101)

param_grid_DT = {
    'max_depth': [2, 8, 32, 128],
    'min_samples_leaf': [1, 3, 9]
}
print("Defined base_tree and param_grid_DT.")

Defined base_tree and param_grid_DT.


In [42]:
# Perform grid search as specified [cite: 270]
searcher_DT = sklearn.model_selection.GridSearchCV(
    base_tree,
    param_grid_DT,
    scoring='balanced_accuracy',
    cv=my_splitter,
    return_train_score=True,
    refit=False, # As specified in PDF [cite: 270]
    n_jobs=-1 # Use all cores
)

print("Starting GridSearchCV for Decision Tree...")
searcher_DT.fit(x_tr_val_NF, y_tr_val_N)
print("GridSearchCV for Decision Tree complete.")

print("\n--- Best Parameters Found ---")
print(searcher_DT.best_params_)
print(f"Best Validation Balanced Accuracy: {searcher_DT.best_score_:.4f}")

Starting GridSearchCV for Decision Tree...
GridSearchCV for Decision Tree complete.

--- Best Parameters Found ---
{'max_depth': 32, 'min_samples_leaf': 3}
Best Validation Balanced Accuracy: 0.7324


In [43]:
print("--- Tabulated Grid Search Results ---")
# Create a DataFrame for easy viewing
results_df = pd.DataFrame(searcher_DT.cv_results_)

# Select and rename columns for clarity
columns_of_interest = [
    'param_max_depth',
    'param_min_samples_leaf',
    'mean_train_score',
    'mean_test_score' # This is the validation score
]
results_df = results_df[columns_of_interest].rename(columns={
    'param_max_depth': 'Max Depth',
    'param_min_samples_leaf': 'Min Samples Leaf',
    'mean_train_score': 'Train BAccuracy',
    'mean_test_score': 'Valid BAccuracy'
})

# Sort by the best validation accuracy
results_col = 'Valid BAccuracy'
results_df = results_df.sort_values(by=results_col, ascending=False)

print(results_df.to_markdown(index=False, floatfmt=".4f"))

print("\n--- Answer to PDF Question ---")
print(f"Q: What are the values of max_depth and min_samples_leaf for the best tree? [cite: 274]")
print(f"A: max_depth = {searcher_DT.best_params_['max_depth']}, "
      f"min_samples_leaf = {searcher_DT.best_params_['min_samples_leaf']}")

--- Tabulated Grid Search Results ---
|   Max Depth |   Min Samples Leaf |   Train BAccuracy |   Valid BAccuracy |
|------------:|-------------------:|------------------:|------------------:|
|     32.0000 |             3.0000 |            0.8772 |            0.7324 |
|    128.0000 |             1.0000 |            0.9962 |            0.7303 |
|     32.0000 |             1.0000 |            0.9182 |            0.7274 |
|    128.0000 |             9.0000 |            0.8323 |            0.7232 |
|    128.0000 |             3.0000 |            0.9242 |            0.7165 |
|     32.0000 |             9.0000 |            0.8150 |            0.7135 |
|      8.0000 |             1.0000 |            0.7265 |            0.6989 |
|      8.0000 |             3.0000 |            0.7219 |            0.6976 |
|      8.0000 |             9.0000 |            0.7128 |            0.6962 |
|      2.0000 |             1.0000 |            0.6404 |            0.6339 |
|      2.0000 |             3.0000 |  

### Build the best decision tree

**TODO Build the Best Tree on the training set** in the next coding cell



In [44]:
# best_tree = base_tree # TODO call set_params using the best_params_ found by your searcher
# best_tree.fit(x_tr_NF, y_tr_N)

# Get the best params from the searcher
best_params_DT = searcher_DT.best_params_
print(f"Best params found: {best_params_DT}")

# Create and fit the best tree
best_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini', min_samples_split=2, random_state=101)
best_tree.set_params(**best_params_DT)

best_tree.fit(x_tr_NF, y_tr_N)
print("Best Tree fitted on the training set.")

Best params found: {'max_depth': 32, 'min_samples_leaf': 3}
Best Tree fitted on the training set.


### Interpret the best decision tree

In [45]:
pretty_print_sklearn_tree(best_tree, feature_names=vocab_list)

The binary tree structure has 817 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    0 are leaves
- depth   4 has   16 nodes, of which    6 are leaves
- depth   5 has   20 nodes, of which    5 are leaves
- depth   6 has   30 nodes, of which   14 are leaves
- depth   7 has   32 nodes, of which   18 are leaves
- depth   8 has   28 nodes, of which   12 are leaves
- depth   9 has   32 nodes, of which   11 are leaves
- depth  10 has   42 nodes, of which   21 are leaves
- depth  11 has   42 nodes, of which   23 are leaves
- depth  12 has   38 nodes, of which   17 are leaves
- depth  13 has   42 nodes, of which   21 are leaves
- depth  14 has   42 nodes, of which   25 are leaves
- depth  15 has   34 nodes, of which   18 are leaves
- depth  16 has   32 nodes, of which   14 are leaves
- depth  17 has   36 nodes, of which   23 are leaves
- dep

In [46]:
print("--- Best Decision Tree Visualization ---")
# Note: This tree might be very large and unreadable if max_depth is 32 or 128
# For a deep tree, we can just print the first few levels
# I'll export_text directly with a max_depth limit for *printing*

print(f"Printing summary of best tree (max_depth={best_params_DT['max_depth']})...")
print("Note: If the tree is large, this will only show the top levels.\n")

print(sklearn.tree.export_text(best_tree, feature_names=vocab_list, max_depth=4))

# If you want to try the full print (might be huge):
# try:
#     pretty_print_sklearn_tree(best_tree, feature_names=vocab_list)
# except Exception as e:
#     print(f"\nCould not print full tree, it is likely too large. Error: {e}")

--- Best Decision Tree Visualization ---
Printing summary of best tree (max_depth=32)...
Note: If the tree is large, this will only show the top levels.

|--- great <= 0.50
|   |--- excel <= 0.50
|   |   |--- disappoint <= 0.50
|   |   |   |--- easy <= 0.50
|   |   |   |   |--- love <= 0.50
|   |   |   |   |   |--- truncated branch of depth 28
|   |   |   |   |--- love >  0.50
|   |   |   |   |   |--- truncated branch of depth 28
|   |   |   |--- easy >  0.50
|   |   |   |   |--- bad <= 0.50
|   |   |   |   |   |--- truncated branch of depth 19
|   |   |   |   |--- bad >  0.50
|   |   |   |   |   |--- truncated branch of depth 3
|   |   |--- disappoint >  0.50
|   |   |   |--- be_disappointed <= 0.50
|   |   |   |   |--- televis <= 0.50
|   |   |   |   |   |--- truncated branch of depth 11
|   |   |   |   |--- televis >  0.50
|   |   |   |   |   |--- class: 1.0
|   |   |   |--- be_disappointed >  0.50
|   |   |   |   |--- the_way <= 0.50
|   |   |   |   |   |--- truncated branch of dep

# Problem 2: Random forest

## 2A: Train a random forest with default settings

In [47]:
simple_forest = sklearn.ensemble.RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_features='sqrt',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101)

### Fit the forest

**TODO Train on the training set** in the next coding cell

In [48]:
# Train on the training set
simple_forest.fit(x_tr_NF, y_tr_N)
print("Simple Random Forest (max_depth=3, n_estimators=100) fitted.")

Simple Random Forest (max_depth=3, n_estimators=100) fitted.


## 2B & Table 2: Feature Importances

### Table 2
**Sample Output** (Feel free to print all words and organize them in any software)

|**Important Words**|**Unimportant Words**|
|:-:|:-:|
|I1 |  U1  |
|I2 |  U2  |
|I3 |  U3  |
|I4 |  U4  |
|I5 |  U5  |
|I6 |  U6  |
|I7 |  U7  |
|I8 |  U8  |
|I9 |  U9  |
|I0 |  U0  |

In [49]:
# Access the feature_importances_ [cite: 279]
importances = simple_forest.feature_importances_

# Create a DataFrame for easy manipulation
feature_df = pd.DataFrame({
    'word': vocab_list,
    'importance': importances
})

# --- Top 10 Important Words --- [cite: 280]
top_10_features = feature_df.sort_values(by='importance', ascending=False).head(10)

# --- 10 Random Near-Zero Importance Words --- [cite: 281]
near_zero_features = feature_df[feature_df['importance'] <= 0.00001]
num_near_zero = len(near_zero_features)

# Sample 10 random words from this group
if num_near_zero >= 10:
    random_10_near_zero = near_zero_features.sample(10, random_state=101)
else:
    # In case there are fewer than 10, just take all of them
    random_10_near_zero = near_zero_features

# --- Create the Two-Panel Table ---
table_data = {
    'Important Words': top_10_features['word'].values,
    'Unimportant Words': random_10_near_zero['word'].values
}
table_df = pd.DataFrame(table_data)

print("--- Feature Importance Table ---")
print(table_df.to_markdown(index=False))

print(f"\nTotal number of eligible 'near-zero' importance terms (<= 0.00001): {num_near_zero}")

--- Feature Importance Table ---
| Important Words   | Unimportant Words   |
|:------------------|:--------------------|
| return            | doing_the           |
| excel             | bought_these        |
| great             | mix                 |
| worst             | not_all             |
| poor              | vision              |
| disappoint        | example_the         |
| your_money        | the_author's        |
| i_love            | abuse               |
| the_best          | went_on             |
| don't             | also_very           |

Total number of eligible 'near-zero' importance terms (<= 0.00001): 7296


## 2C: Best Random Forest via grid search



This block might take 2-10 minutes. 

If yours runs significantly longer, try this out on Google Colab instead.

In [50]:

# Define parameter grid for Random Forest
param_grid_RF = {
    'max_features': [3, 10, 33, 100, 333],
    'max_depth': [16, 32],
    'min_samples_leaf': [1],
    'n_estimators': [100],
    'random_state': [101]
}
print("Defined param_grid_RF:")
print(param_grid_RF)

Defined param_grid_RF:
{'max_features': [3, 10, 33, 100, 333], 'max_depth': [16, 32], 'min_samples_leaf': [1], 'n_estimators': [100], 'random_state': [101]}


In [51]:
# Define base forest estimator
base_forest = sklearn.ensemble.RandomForestClassifier(
    criterion='gini',
    n_estimators=100,
    random_state=101
)
print("Defined base_forest estimator.")

Defined base_forest estimator.


In [52]:
# Define the searcher object
searcher_RF = sklearn.model_selection.GridSearchCV(
    base_forest,
    param_grid_RF,
    scoring='balanced_accuracy',
    cv=my_splitter,
    return_train_score=True,
    refit=False, # As specified in PDF for Decision Tree
    n_jobs=-1 # Use all cores
)
print("Defined searcher_RF (GridSearchCV object).")

Defined searcher_RF (GridSearchCV object).


### Do the search!

In [53]:
print("Starting GridSearchCV for Random Forest...")
start_time = time.time()

searcher_RF.fit(x_tr_val_NF, y_tr_val_N)

end_time = time.time()
print(f"GridSearchCV for Random Forest complete. Took {end_time - start_time:.2f} seconds.")

Starting GridSearchCV for Random Forest...
GridSearchCV for Random Forest complete. Took 41.16 seconds.


### Display search results

In [54]:
print("--- Tabulated Grid Search Results (Random Forest) ---")
# Create a DataFrame for easy viewing
results_rf_df = pd.DataFrame(searcher_RF.cv_results_)

# Select and rename columns for clarity
columns_of_interest_rf = [
    'param_max_depth',
    'param_max_features',
    'mean_train_score',
    'mean_test_score' # This is the validation score
]
results_rf_df = results_rf_df[columns_of_interest_rf].rename(columns={
    'param_max_depth': 'Max Depth',
    'param_max_features': 'Max Features',
    'mean_train_score': 'Train BAccuracy',
    'mean_test_score': 'Valid BAccuracy'
})

# Sort by the best validation accuracy
results_rf_col = 'Valid BAccuracy'
results_rf_df = results_rf_df.sort_values(by=results_rf_col, ascending=False)

print(results_rf_df.to_markdown(index=False, floatfmt=".4f"))

print("\n--- Best Model Parameters ---")
print(f"Best parameters found: {searcher_RF.best_params_}")

print("\n--- Analytical Questions ---")
print(f"Q: What is the value of max_features of your best forest?")
print(f"A: {searcher_RF.best_params_['max_features']}")

print(f"\nQ: What is the maximum possible value for max_features for this dataset?")
print(f"A: The maximum possible value is {x_tr_NF.shape[1]} (the total number of features).")

print(f"\nQ: Why is it beneficial to tune this hyperparameter?")
print("A: Tuning `max_features` is crucial for Random Forests. Setting it to a value less "
      "than the total number of features *decorrelates* the trees. "
      "If all trees saw all features, they would likely all pick the same "
      "few 'super-predictive' features at the top, making all the trees "
      "very similar. By forcing each tree to choose from a random subset "
      "of features at each split, we ensure the trees are different, "
      "which reduces the variance of the ensemble and improves generalization.")

print(f"\nQ: When fitting random forests, what is the primary tradeoff "
      "controlled by the n_estimators hyperparameter?")
print("A: The primary tradeoff is between *performance/stability* and *computational cost*. "
      "A higher `n_estimators` (more trees) generally leads to a more stable "
      "and better-performing model (up to a point) because the variance of the "
      "ensemble's prediction is reduced. However, this comes at a linear "
      "increase in training time and memory.")

print("A: No, you generally cannot overfit by setting `n_estimators` too large. "
      "This is a fundamental benefit of Random Forests. Here's why:\n"
      "\n"
      "Mathematical Intuition:\n"
      "- For a Random Forest ensemble: f̂(x) = (1/B) Σ f̂_b(x), where B = n_estimators\n"
      "- As B increases: Variance ≈ σ²/B (decreases), while Bias remains constant\n"
      "- The error converges to a lower limit and plateaus, never increasing\n"
      "\n"
      "Conceptual Explanation:\n"
      "Each tree is trained independently on a bootstrapped sample. Adding more trees "
      "averages in more independent predictions, which stabilizes the ensemble and "
      "reduces variance. Unlike increasing the depth of individual trees (which can "
      "cause overfitting), adding more trees just makes the predictions more stable.\n"
      "\n"
      "Practical Outcome:\n"
      "- Training accuracy may increase slightly, then plateau\n"
      "- Validation/test accuracy improves, then plateaus (doesn't decrease)\n"
      "- The only tradeoff is computational cost (time and memory)\n"
      "\n"
      "Note: Individual trees can still overfit if too deep, but the ensemble's "
      "aggregation prevents overall overfitting as n_estimators increases.")

--- Tabulated Grid Search Results (Random Forest) ---
|   Max Depth |   Max Features |   Train BAccuracy |   Valid BAccuracy |
|------------:|---------------:|------------------:|------------------:|
|     32.0000 |        33.0000 |            0.9644 |            0.8512 |
|     32.0000 |        10.0000 |            0.9701 |            0.8436 |
|     16.0000 |        33.0000 |            0.9236 |            0.8424 |
|     16.0000 |        10.0000 |            0.9250 |            0.8410 |
|     16.0000 |       100.0000 |            0.9207 |            0.8394 |
|     32.0000 |       100.0000 |            0.9697 |            0.8354 |
|     32.0000 |       333.0000 |            0.9527 |            0.8122 |
|     32.0000 |         3.0000 |            0.9682 |            0.7965 |
|     16.0000 |       333.0000 |            0.8712 |            0.7842 |
|     16.0000 |         3.0000 |            0.9083 |            0.7518 |

--- Best Model Parameters ---
Best parameters found: {'max_depth': 32

### Build the best random forest using the best hyperparameters found in 2B 

This is necessary so you have the specific best performing forest in your workspace.

Train *only* on training set (do not merge train and valid)


In [55]:
# Get the best params from the searcher
best_params_RF = searcher_RF.best_params_
print(f"Best params found: {best_params_RF}")

# Create and fit the best forest
# We need to make sure all params from the grid search are set
best_forest = sklearn.ensemble.RandomForestClassifier(
    criterion='gini'
)
best_forest.set_params(**best_params_RF)

best_forest.fit(x_tr_NF, y_tr_N)
print("Best Random Forest fitted on the training set.")

Best params found: {'max_depth': 32, 'max_features': 33, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
Best Random Forest fitted on the training set.


### Table 3: Comparison of methods on the bag-of-words to sentiment classification task.

Please report **balanced accuracy** on the train, valid, and test sets, to 3 digits of precision

**Sample Output** (Feel free to print all values and organize them by hand)

|**method**|**max depth**|**num trees**|**train BAcc**|**valid BAcc**|**test BAcc**|
|:-|:-:|:-:|:-:|:-:|:-:|
|simple Tree	| 1 | 1 | 0.123	|0.456	|0.890|
|best Tree	|1 | 1 | 0.123	|0.456	|0.890|
|simple RandomForest	|1 | 1 | 0.123	|0.456	|0.890|
|best RandomForest	|1 | 1 | 0.123	|0.456	|0.890|

In [56]:
print("--- Calculating Final Metrics for Table 3 ---")

models = {
    'Simple Tree': simple_tree,
    'Best Tree': best_tree,
    'Simple RandomForest': simple_forest,
    'Best RandomForest': best_forest
}

data_splits = {
    'train': (x_tr_NF, y_tr_N),
    'valid': (x_va_NF, y_va_N),
    'test': (x_te_NF, y_te_N)
}

# Calculate metrics for all models
table_3_data = []
for name, model in models.items():
    row = {
        'method': name,
        'max depth': model.get_params()['max_depth'],
        'num trees': model.get_params().get('n_estimators', 1) # DTs have 1 tree
    }
    
    for split_name, (X, y) in data_splits.items():
        y_pred = model.predict(X)
        bacc = sklearn.metrics.balanced_accuracy_score(y, y_pred)
        row[f'{split_name} BAcc'] = bacc
    
    table_3_data.append(row)

# Create and print the DataFrame
table_3_df = pd.DataFrame(table_3_data)

print("\n--- Table 3: Comparison of Methods ---")
print(table_3_df.to_markdown(index=False, floatfmt=".3f"))

print("\n--- Summary Conclusions ---")
print("Based on the (random) placeholder data:")
print("1. The 'Best Tree' (after tuning) likely shows much higher training accuracy "
      "than the 'Simple Tree', indicating it has learned the training data "
      "more closely (or overfit).")
print("2. The 'Simple RandomForest' likely has better validation/test accuracy "
      "than the 'Simple Tree' of the same depth, showing the power of ensembling.")
print("3. The 'Best RandomForest' (tuned) is expected to have the highest "
      "validation and test accuracy, demonstrating the combined benefit of "
      "ensembling and hyperparameter tuning.")


--- Calculating Final Metrics for Table 3 ---

--- Table 3: Comparison of Methods ---
| method              |   max depth |   num trees |   train BAcc |   valid BAcc |   test BAcc |
|:--------------------|------------:|------------:|-------------:|-------------:|------------:|
| Simple Tree         |           3 |           1 |        0.646 |        0.645 |       0.646 |
| Best Tree           |          32 |           1 |        0.877 |        0.732 |       0.749 |
| Simple RandomForest |           3 |         100 |        0.819 |        0.797 |       0.778 |
| Best RandomForest   |          32 |         100 |        0.964 |        0.851 |       0.837 |

--- Summary Conclusions ---
Based on the (random) placeholder data:
1. The 'Best Tree' (after tuning) likely shows much higher training accuracy than the 'Simple Tree', indicating it has learned the training data more closely (or overfit).
2. The 'Simple RandomForest' likely has better validation/test accuracy than the 'Simple Tree' of